In [1]:
!pip install -q tensorflow-datasets importlib-resources

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model

print("All libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

All libraries imported successfully!
TensorFlow version: 2.19.0


In [3]:
(train_raw, val_raw), info = tfds.load(
    'oxford_iiit_pet',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True
)

print("Dataset downloaded successfully!")
print("Training images:", info.splits['train'].num_examples)

Dl Completed...: 100% 1/1 [00:46<00:00, 46.15s/ url]
Dl Size...: 100% 773/773 [00:46<00:00, 16.75 MiB/s]
Extraction completed...: 100% 2/2 [00:11<00:00, 5.71s/ file]
Generating splits...: 100% 2/2 [00:40<00:00, 20.16s/ splits]
Generating train examples...: 100% 3680/3680 [00:26<00:00, 138.68 examples/s]
Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.8K545Z_4.0.0/oxford_iiit_pet-train.tfrecord*...: 100% 3680/3680 [00:00<00:00, 6235.14 examples/s]
Generating test examples...: 100% 3669/3669 [00:13<00:00, 272.24 examples/s]
Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.8K545Z_4.0.0/oxford_iiit_pet-test.tfrecord*...: 100% 3669/3669 [00:00<00:00, 5672.14 examples/s]
Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


Dataset downloaded successfully!
Training images: 3680


In [4]:
IMG_SIZE = 128
BATCH_SIZE = 8

def preprocess(data):
    image = tf.image.resize(
        data['image'],
        (IMG_SIZE, IMG_SIZE)
    )
    mask = tf.image.resize(
        data['segmentation_mask'],
        (IMG_SIZE, IMG_SIZE),
        method='nearest'
    )
    image = tf.cast(image, tf.float32) / 255.0
    mask = tf.cast(mask, tf.int32) - 1
    mask = tf.squeeze(mask, axis=-1)
    return image, mask

train_data = train_raw.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
val_data = val_raw.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_data = train_data.shuffle(300).batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)
val_data = val_data.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

print("Preprocessing completed successfully!")

Preprocessing completed successfully!


In [5]:
for images, masks in train_data.take(1):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(images[0])
    plt.title("Original Image")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(masks[0], cmap="viridis")
    plt.title("Segmentation Mask")
    plt.axis("off")
    plt.show()

[Matplotlib Figure: Original Image vs Segmentation Mask]


In [6]:
def unet_model():
    inputs = layers.Input((128, 128, 3))
    # Encoder
    c1 = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(16, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(32, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(32, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b = layers.Conv2D(128, 3, activation='relu', padding='same')(p3)

    # Decoder
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(b)
    u1 = layers.Concatenate()([u1, c3])
    d1 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)

    u2 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(d1)
    u2 = layers.Concatenate()([u2, c2])
    d2 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)

    u3 = layers.Conv2DTranspose(16, 2, strides=2, padding='same')(d2)
    u3 = layers.Concatenate()([u3, c1])
    d3 = layers.Conv2D(16, 3, activation='relu', padding='same')(u3)

    outputs = layers.Conv2D(3, 1, activation='softmax')(d3)
    return Model(inputs, outputs)

unet = unet_model()
print("UNet model created successfully!")
print("Output shape:", unet.output_shape)

UNet model created successfully!
Output shape: (None, 128, 128, 3)


In [7]:
unet.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
unet_history = unet.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

Epoch 1/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 166s 437ms/step - accuracy: 0.6789 - loss: 0.7615 - val_accuracy: 0.7285 - val_loss: 0.6592
Epoch 2/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 161s 437ms/step - accuracy: 0.7510 - loss: 0.6143 - val_accuracy: 0.7690 - val_loss: 0.5692
Epoch 3/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 162s 441ms/step - accuracy: 0.7850 - loss: 0.5382 - val_accuracy: 0.7796 - val_loss: 0.5476


In [8]:
def conv_block(x, filters):
    x = layers.Conv2D(
        filters, 3, padding='same', activation='relu'
    )(x)
    x = layers.Conv2D(
        filters, 3, padding='same', activation='relu'
    )(x)
    return x

def attention_gate(skip, gate, filters):
    theta = layers.Conv2D(filters, 1, padding='same')(skip)
    phi = layers.Conv2D(filters, 1, padding='same')(gate)
    x = layers.Add()([theta, phi])
    x = layers.Activation('relu')(x)
    psi = layers.Conv2D(
        1, 1, activation='sigmoid'
    )(x)
    return layers.Multiply()([skip, psi])

def attention_unet():
    inputs = layers.Input((128, 128, 3))
    # Encoder
    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 64)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b = conv_block(p3, 128)

    # Decoder
    u1 = layers.Conv2DTranspose(
        64, 2, strides=2, padding='same'
    )(b)
    a1 = attention_gate(c3, u1, 64)
    u1 = layers.Concatenate()([u1, a1])
    d1 = conv_block(u1, 64)

    u2 = layers.Conv2DTranspose(
        32, 2, strides=2, padding='same'
    )(d1)
    a2 = attention_gate(c2, u2, 32)
    u2 = layers.Concatenate()([u2, a2])
    d2 = conv_block(u2, 32)

    u3 = layers.Conv2DTranspose(
        16, 2, strides=2, padding='same'
    )(d2)
    a3 = attention_gate(c1, u3, 16)
    u3 = layers.Concatenate()([u3, a3])
    d3 = conv_block(u3, 16)

    outputs = layers.Conv2D(
        3, 1, activation='softmax'
    )(d3)
    return Model(inputs, outputs)

attention_model = attention_unet()
print("Attention UNet created successfully!")
print("Output shape:", attention_model.output_shape)

Attention UNet created successfully!
Output shape: (None, 128, 128, 3)


In [9]:
attention_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
attention_history = attention_model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

Epoch 1/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 237s 630ms/step - accuracy: 0.6776 - loss: 0.7524 - val_accuracy: 0.7209 - val_loss: 0.6680
Epoch 2/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 231s 628ms/step - accuracy: 0.7615 - loss: 0.5947 - val_accuracy: 0.7931 - val_loss: 0.5206
Epoch 3/3
368/368 ━━━━━━━━━━━━━━━━━━━━ 229s 621ms/step - accuracy: 0.7975 - loss: 0.5117 - val_accuracy: 0.7966 - val_loss: 0.5105


In [10]:
for images, masks in val_data.take(1):
    prediction = unet.predict(images, verbose=0)
    predicted_mask = np.argmax(prediction[0], axis=-1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(images[0])
    plt.title("Original Image")
    plt.axis("off")
    plt.subplot(1, 3, 2)
    plt.imshow(masks[0], cmap='viridis')
    plt.title("Actual Mask")
    plt.axis("off")
    plt.subplot(1, 3, 3)
    plt.imshow(predicted_mask, cmap='viridis')
    plt.title("UNet Prediction")
    plt.axis("off")
    plt.show()

[Matplotlib Figure: Original Image, Actual Mask, UNet Prediction]


In [11]:
plt.figure(figsize=(8, 5))
plt.plot(
    unet_history.history['accuracy'],
    marker='o',
    label='UNet'
)
plt.plot(
    attention_history.history['accuracy'],
    marker='o',
    label='Attention UNet'
)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("UNet vs Attention UNet")
plt.legend()
plt.grid()
plt.show()

[Matplotlib Figure: UNet vs Attention UNet Accuracy Plot]
